# 01 The extraction: what the study reads, checked before anything is read

**What this notebook does.** It checks that the CRSP and Compustat files on this machine are the extraction the study is built on, lists the eight tables the study reads with their rows, fetch dates and hashes, checks that every column the later notebooks read is there, and checks the code copied from two other studies. It reads no row of any table. Everything it prints comes from the extraction's manifest, from each file's own metadata and from the copied files.

**Why it comes first.** Every number in the later notebooks is a statement about these files. If a file had changed since it was fetched, every later number would describe another extraction. The checks take about a minute and run before anything else.

**What it judges.** No gate and no expectation. A file that differs from the manifest stops the study here.

**The data.** CRSP and Compustat, from Wharton Research Data Services, accessed through Queen Mary University of London, extracted on 28 September 2026. The extraction stays on the machine that holds it and is never committed.

## Terms used in this notebook

| Term | Meaning |
|---|---|
| extraction | the copy of CRSP and Compustat tables taken on 28 September 2026, kept in the folder named by the environment variable `MLRP_DATA_ROOT` on the machine that holds it |
| manifest | the extraction's own record. For every data file it lists the query that produced it, its rows and columns, its size, its SHA-256 hash and the time it was fetched |
| SHA-256 hash | a fingerprint of 64 characters computed from a file's bytes. A change of one byte changes the hash |
| digest of the entries | the SHA-256 of the manifest's records for the study's eight tables alone: each file's name, rows and hash. A pull that adds other tables to the extraction changes the manifest's hash and leaves this digest as it is |
| table | one of the extraction's datasets, such as CRSP's monthly stock file. A table is stored in one file or in one file per year |
| copied code | seven modules of the study Stock-Level Tracking and one of the study Signal Evaluation, copied unchanged into `src/mlrp/source/` |
| Sharpe ratio | a portfolio's mean return over the standard deviation of its return |
| provenance record | `results/20260928/provenance.json`: the extraction, its two checks, the fetch dates, the commits the copied code comes from and the versions of the tools every later result was built with |

## What I expect to see, and where it comes from

1. The manifest's SHA-256 begins `23e96d18`, and it lists 310 files and 112,161,796 rows: the values in `src/mlrp/constants.py`, read from the manifest on 18 November 2026.
2. The study reads eight tables in 95 files: CRSP's list of index members, its monthly index file, its monthly and daily stock files, its delistings, its names, its link to Compustat, and Compustat's annual fundamentals. Together they hold 73,595,093 rows, all fetched on 28 September 2026. The count of files is `MANIFEST_ENTRIES` in the constants, and the count of rows `MANIFEST_ENTRIES_ROWS`, the sum of the manifest's counts for the eight tables, read on 21 November 2026.
3. Every file's SHA-256 on disk equals the manifest's, and every file's row count in its own footer equals the manifest's.
4. Every column the later notebooks read is in every file of its table.
5. The SHA-256 of each copied file equals the value in the constants.

Running time: about a minute, most of it spent computing the hash of the daily stock file, 2.2 GB in 37 yearly files.

In [ ]:
import hashlib
import json
import pathlib
import sys
import time

# The package lives in src/ beside this folder. The notebook finds the repository's root from where it runs.
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "mlrp").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from mlrp import constants as C
from mlrp import data as D
from mlrp.source import extraction as X
from mlrp.source import layer as L
from mlrp.source import returns as R
from mlrp.source import universe as U

# Stops with one sentence when the data root is not set, when the manifest is not the one the constants fix,
# or when one of the study's eight tables is not the one they record.
root, manifest = D.open_extraction()
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}", "| entries of the eight tables", C.MANIFEST_ENTRIES_SHA256[:12])

## 1. The tables the study reads

The setup cell has already run two checks: the manifest's SHA-256 against the constants, and the digest of the eight tables' entries. Each row below is one table: the files it is stored in, its rows and columns, and the date it was fetched, in Amsterdam time.

In [ ]:
records = D.files_used(manifest)
files = pd.DataFrame(records)
by_table = (files.groupby(["key", "table"], as_index=False)
                 .agg(files=("file", "size"), rows=("rows", "sum"), columns=("columns", "first"), fetch_date=("fetch_date", "max")))
print(by_table.to_string(index=False))
print()
print("tables", len(by_table), "| files", len(files), "| rows", f"{files['rows'].sum():,}")

## 2. The files on disk against the manifest

Two checks for each file, each independent of the other. The SHA-256 of the file on disk is computed again and compared with the manifest's. The row count in the file's footer, which the parquet format writes when the file is saved, is compared with the count in the manifest. A file that fails either check stops the study here.

In [ ]:
t0 = time.time()
checks = pd.DataFrame(X.verify_files(root, records))
print(f"checked {len(checks)} files in {time.time() - t0:.0f} s")
print("present", int(checks["exists"].sum()), "| hash equal to the manifest's", int(checks["sha256_matches"].sum()),
      "| row count equal to the manifest's", int(checks["rows_match"].sum()))
failed = checks[~(checks["exists"] & checks["sha256_matches"] & checks["rows_match"])]
if len(failed):
    print(failed.to_string(index=False))
    raise SystemExit("At least one file differs from the manifest, so the study stops here.")

## 3. The columns the later notebooks read

Each later notebook reads only the columns it needs, listed in `src/mlrp/data.py` as `COLUMNS`: the copied lists for the seven CRSP tables, the monthly share volume added for dollar volume, and the company key, the fiscal year end, the fiscal year and fifteen items of Compustat's annual fundamentals. The schema of every file is checked here, once.

In [ ]:
lacking = D.missing_columns(root, records)
print("files lacking a column a later notebook reads:", len(lacking))
if lacking:
    print(lacking)
    raise SystemExit("A column a later notebook reads is missing, so the study stops here.")
print({key: len(cols) for key, cols in D.COLUMNS.items()})

## 4. The copied code

The universe, the returns and the reading of the extraction come from Stock-Level Tracking's data layer, copied unchanged from its commit named in the constants, and the interval for the difference of two Sharpe ratios comes from Signal Evaluation. The SHA-256 of each copy is computed here and compared with the constants. The tests compare each copy with its source file as well.

In [ ]:
source = ROOT / C.COPIED_INTO
copies = [(name, path, digest) for name, (path, digest) in C.COPIED_FILES.items()]
copies.append((pathlib.PurePosixPath(C.BOOTSTRAP_FILE).name, C.BOOTSTRAP_FILE, C.BOOTSTRAP_SHA256))
table = pd.DataFrame([{"file": name, "from": path, "sha256": hashlib.sha256((source / name).read_bytes()).hexdigest()[:16],
                       "equal_to_the_constants": hashlib.sha256((source / name).read_bytes()).hexdigest() == digest}
                      for name, path, digest in copies])
print(table.to_string(index=False))
print("copied from", C.SOURCE_REPOSITORY, C.SOURCE_COMMIT[:7], "and", C.BOOTSTRAP_SOURCE_REPOSITORY, C.BOOTSTRAP_SOURCE_COMMIT)
if not table["equal_to_the_constants"].all():
    raise SystemExit("A copied file differs from the constants, so the study stops here.")

## 5. What is committed

Two files go into `results/20260928/`. `extraction_files.csv` has one row per file used: its key, table, path, the query that produced it, its rows, columns and size, its SHA-256 and its fetch date. It is the manifest's metadata and holds no value from inside any file. A reader with access to Wharton Research Data Services can run the same queries and compare the hashes. `provenance.json` is the provenance record that every later result names.

In [ ]:
committed = files[["key", "table", "file", "query", "rows", "columns", "bytes", "sha256", "fetch_date"]]
p1 = D.write_results("extraction_files.csv", frame=committed)
record = D.provenance(manifest, records, written_by="notebooks/01_extraction.ipynb")
p2 = D.write_results("provenance.json", record=record)
print("written:", p1.relative_to(ROOT), "and", p2.relative_to(ROOT))
summary = pd.DataFrame([
    ("manifest SHA-256 and the entries' digest", "equal to the constants", "equal (the setup cell stops otherwise)"),
    ("tables, files, rows", f"{C.TABLE_COUNT}, {C.MANIFEST_ENTRIES}, {C.MANIFEST_ENTRIES_ROWS:,}", f"{len(by_table)}, {len(files)}, {files['rows'].sum():,}"),
    ("files equal to the manifest", "95 of 95", f"{int((checks['sha256_matches'] & checks['rows_match']).sum())} of {len(checks)}"),
    ("files lacking a column", "0", str(len(lacking))),
    ("copies equal to the constants", "8 of 8", f"{int(table['equal_to_the_constants'].sum())} of {len(table)}"),
], columns=["check", "expected", "result"])
print(summary.to_string(index=False))

## What was learned, and what could be wrong

- The table above sets each expectation of this notebook beside its result.
- A hash shows that a file has not changed since it was fetched. It does not show that the file holds what the vendor intended. Notebook 02 tests the universe and the returns against Stock-Level Tracking's counts, which were themselves tested against CRSP's own counts and returns.
- CRSP corrects its history with each release of its data, so the same queries run later can return other files, with other hashes, for reasons that are not errors.
- The row counts in the manifest and in the footers are both counts of rows as stored. Neither says anything about a value inside a row. Notebooks 03 and 04 count the missing values of each characteristic by year.